# Step 1 · Hello, local LLM
One prompt to a model running **on this machine**, sent through the **Bifrost** gateway.
Nothing leaves the laptop.

> **Standalone.** All the logic is in the cells below and the demo data is in `data/`. Nothing is imported from the app.
> It only needs a model gateway: start the gateway first (SETUP.md, section 5: Bifrost at `http://127.0.0.1:8080/v1`).
> Install what it needs with `pip install -r requirements.txt`.

## 1. Ask for a *role*, never a model
The code below asks for `analyst`, `judge` or `embed`. Only the `MODELS` map names a model, so swapping one is a one-line change.

In [1]:
# Everything this notebook needs is in this file and in data/. It imports nothing from copilot/.
import os
os.environ.update(ANONYMIZED_TELEMETRY="False", RAGAS_DO_NOT_TRACK="true", LANGSMITH_TRACING="false",
                  LANGCHAIN_TRACING_V2="false", HF_HUB_OFFLINE="1")   # nothing leaves the machine

import time
from openai import OpenAI

GATEWAY_URL = os.getenv("BIFROST_URL", "http://127.0.0.1:8080/v1")     # Bifrost, running on this machine
MODELS = {"analyst": "ollama/qwen2.5:7b-instruct",     # the role -> model map: code asks for a role
          "judge":   "ollama/llama3.1:8b",
          "embed":   "ollama/nomic-embed-text"}
client = OpenAI(base_url=GATEWAY_URL, api_key="local") # a local gateway needs no real key, but the SDK wants a value


class GatewayError(RuntimeError):
    """A model call failed. There is no fallback model: fail loudly."""


def chat(role, prompt, json_mode=False, max_tokens=700):
    """Send one prompt to the model behind a role. Temperature 0: as repeatable as a model gets."""
    kwargs = {"response_format": {"type": "json_object"}} if json_mode else {}
    try:
        reply = client.chat.completions.create(model=MODELS[role], temperature=0, max_tokens=max_tokens, timeout=900,
                                               messages=[{"role": "user", "content": prompt}], **kwargs)
    except Exception as exc:
        raise GatewayError(f"Model role '{role}' failed: {exc}") from exc
    return (reply.choices[0].message.content or "").strip()


def embed(texts):
    """Turn texts into vectors with the embed role."""
    try:
        return [d.embedding for d in client.embeddings.create(model=MODELS["embed"], input=texts).data]
    except Exception as exc:
        raise GatewayError(f"Model role 'embed' failed: {exc}") from exc


print("Gateway:", GATEWAY_URL)

Gateway: http://127.0.0.1:8080/v1


## 2. Send one prompt through the gateway

In [2]:
start = time.perf_counter()
reply = chat("analyst", "In one sentence: what does a payroll provider do for its clients?")
print(reply)
print(f"\n{time.perf_counter() - start:.1f} seconds (the first call also loads the model into memory)")

A payroll provider manages the calculation, processing, and distribution of employees' wages, including deductions and tax withholdings, for its clients.

3.3 seconds (the first call also loads the model into memory)


## 3. How fast is it on this machine?
The same request again, now the model is loaded, reading the token counts the gateway reports.

In [3]:
PROMPT = "In three sentences, explain what a limitation of liability clause does in a contract."
for run in range(1, 4):
    start = time.perf_counter()
    response = client.chat.completions.create(model=MODELS["analyst"], temperature=0, max_tokens=120,
                                              messages=[{"role": "user", "content": PROMPT}])
    seconds = time.perf_counter() - start
    tokens = response.usage.completion_tokens
    print(f"run {run}: {seconds:5.1f}s  {tokens} tokens  {tokens / seconds:4.1f} tokens/second")

run 1:   3.8s  75 tokens  19.9 tokens/second


run 2:   3.0s  65 tokens  21.6 tokens/second


run 3:   3.0s  65 tokens  21.8 tokens/second


## 4. Embeddings: text becomes numbers
Similar meaning gives similar numbers. This is what makes searching a contract by meaning possible.

In [4]:
import math

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b)) / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

start = time.perf_counter()
clause, close, far = embed(["The provider's liability is capped at twelve months of fees.",
                            "How much do we pay at most if something goes wrong?",
                            "The cafeteria serves lunch from noon."])
print(f"{len(clause)} numbers per text, 3 texts in {time.perf_counter() - start:.2f}s")
print(f"clause vs related question : {cosine(clause, close):.3f}")
print(f"clause vs unrelated text   : {cosine(clause, far):.3f}")

768 numbers per text, 3 texts in 1.13s
clause vs related question : 0.538
clause vs unrelated text   : 0.348


## 5. The gateway refuses anything not on its allowlist
Bifrost only serves the three local models. A different local model and a cloud model are both refused, and the code fails loudly instead of quietly using something else.

In [5]:
for role, model in (("not on the allowlist", "ollama/llama3.2:1b"), ("a cloud model", "openai/gpt-4o")):
    MODELS["test"] = model
    try:
        chat("test", "hi")
        print(f"{role}: ANSWERED (this should not happen)")
    except GatewayError as exc:
        print(f"{role}: refused\n   {str(exc)[:150]}")
del MODELS["test"]

not on the allowlist: refused
   Model role 'test' failed: Error code: 400 - {'type': 'no_key_supports_model', 'is_bifrost_error': False, 'status_code': 400, 'error': {'error': 'no ke
a cloud model: refused
   Model role 'test' failed: Error code: 400 - {'is_bifrost_error': False, 'error': {'error': 'failed to get config for provider openai: not found', 'mes


**Takeaway:** one door to the models, every call visible, and nothing outside the allowlist can run.